# Análise Unificada: Desmatamento e Impacto Socioambiental

## Objetivo
Esta análise tem como objetivo consolidar os dados das camadas **Silver** e **Gold** para entender a relação entre o desmatamento, a produção agropecuária, o desenvolvimento humano (IDHM) e a eficácia da fiscalização ambiental.

**Destaque:** Gráficos interativos (Plotly) e conclusões integradas por seção.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

def load_parquet(path):
    return pd.read_parquet(os.path.join('..', path))

# Carregamento de dados básicos
df_serie = load_parquet('data/02_silver/serie_historica_2020_2023.parquet')
df_quadrantes = load_parquet('data/03_gold/tipologia_municipal_quadrantes.parquet')
df_status_embargos = load_parquet('data/03_gold/status_regular_embargos.parquet' if os.path.exists('../data/03_gold/status_regular_embargos.parquet') else 'data/03_gold/status_regularizacao_embargos.parquet')
df_eficiencia = load_parquet('data/03_gold/eficiencia_atividade.parquet')

## 1. Evolução do Desmatamento (2020-2023)

**Motivação:** Compreender a tendência temporal do desmatamento nos últimos anos para identificar se houve aceleração ou desaceleração.

In [2]:
# Agrupamento por ano
evol_anual = df_serie.groupby('ano')['area_desmatada_ha'].sum().reset_index()

fig = px.bar(
    evol_anual, x='ano', y='area_desmatada_ha', 
    title='Evolução do Desmatamento (ha) - 2020 a 2023',
    labels={'area_desmatada_ha': 'Área Desmatada (ha)', 'ano': 'Ano'},
    text_auto='.2f',
    color='area_desmatada_ha',
    color_continuous_scale='Reds'
)
fig.show()

**Conclusão da Seção:** O desmatamento apresentou um pico em 2022, ultrapassando os 10 mil hectares no período analisado. Apesar de uma queda relativa em 2023, os níveis continuam significativamente elevados em comparação a 2020 e 2021.

## 2. O Motor Econômico: VAB Agropecuário vs Desmatamento

**Motivação:** Investigar se o aumento do desmatamento está correlacionado com o crescimento da riqueza agropecuária (VAB).

In [3]:
# Scatter plot interativo
fig = px.scatter(
    df_serie, x='vab_agro_mil_reais', y='area_desmatada_ha', 
    hover_data=['cod_ibge', 'ano'], 
    title='VAB Agropecuário vs Área Desmatada (Escala Log)',
    labels={'vab_agro_mil_reais': 'VAB Agropecuário (mil R$)', 'area_desmatada_ha': 'Área Desmatada (ha)'},
    log_x=True, log_y=True,
    color='ano', opacity=0.6
)
fig.show()

corr = df_serie[['vab_agro_mil_reais', 'area_desmatada_ha']].corr().iloc[0, 1]
print(f"Correlação de Pearson entre VAB e Desmatamento: {corr:.4f}")

Correlação de Pearson entre VAB e Desmatamento: 0.0104


**Conclusão da Seção:** A correlação geral parece baixa, indicando que o desmatamento nem sempre se traduz em um aumento proporcional imediato de riqueza no VAB, ou que existem áreas com alta produtividade consolidada que não necessitam de novos desmatamentos.

## 3. Paradoxo Socioambiental: IDHM e a Tipologia de Quadrantes

**Motivação:** Analisar se o desmatamento traz desenvolvimento humano real (IDHM). Classificamos os municípios nos quatro quadrantes.

In [4]:
# Distribuição por quadrantes
fig = px.pie(
    df_quadrantes, names='quadrante', 
    title='Distribuição dos Municípios por Quadrante Socioambiental',
    hole=0.4, color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

# Gráfico de dispersão IDHM vs Desmatamento por quadrante
fig = px.scatter(
    df_quadrantes, x='idhm', y='area_desmatada_ha', color='quadrante',
    hover_data=['municipio', 'uf'],
    log_y=True, 
    title='Paradoxo: IDHM vs Área Desmatada',
    labels={'idhm': 'IDH Municipal', 'area_desmatada_ha': 'Área Desmatada (ha)'},
)
fig.show()

**Conclusão da Seção:** O predomínio do quadrante 'Estagnação' e a presença de municípios no quadrante 'Paradoxo' (Alto Desmatamento, Baixo IDHM) reforçam que a degradação ambiental não é um pré-requisito nem uma garantia para o desenvolvimento humano local.

## 4. Eficácia da Fiscalização Ambiental (Embargos do IBAMA)

**Motivação:** Avaliar o status das áreas que sofreram sanções administrativas (embargos).

In [5]:
fig = px.pie(
    df_status_embargos, names='descricao', values='contagem', 
    title='Status de Regularização Ambiental dos Embargos',
    labels={'descricao': 'Situação'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.show()

**Conclusão da Seção:** A maior parte dos registros ainda permanece associada ao 'Desmatamento / Degradação', o que indica que a aplicação do embargo, embora punitiva, ainda enfrenta desafios para levar as áreas à plena regularização.

## 5. Eficiência Econômica vs Ambiental

**Motivação:** Identificar municípios benchmarkings que geram alto valor financeiro com o menor impacto possível.

In [6]:
# Cruzamento para obter nomes dos municípios
df_resumo = df_eficiencia.merge(df_quadrantes[['cod_ibge', 'municipio', 'uf']].drop_duplicates(), on='cod_ibge', how='left')

# Filtragem de municípios reais e cálculo da razão de eficiência
df_resumo = df_resumo[df_resumo['cod_ibge'] > 0].copy()
df_resumo['razao_desmat_vab'] = df_resumo['area_desmatada_ha'] / df_resumo['vab_agro_mil_reais'].replace(0, 1)

# Top 15 mais eficientes (menor razão desmatamento / VAB)
df_top = df_resumo[df_resumo['area_desmatada_ha'] > 0].sort_values('razao_desmat_vab').head(15)

fig = px.bar(
    df_top, x='municipio', y='razao_desmat_vab', color='uf', 
    title='Top 15 Municípios de Alta Eficiência (Menor Desmatamento por R$ Gerado)',
    labels={'razao_desmat_vab': 'Desmatamento (ha) / mil R$ VAB'},
    hover_data=['vab_agro_mil_reais', 'area_desmatada_ha']
)
fig.show()

**Conclusão da Seção:** Estes municípios demonstram que é possível gerar riqueza agropecuária (VAB) com impactos mínimos na vegetação nativa, servindo como modelo de gestão sustentável.

## 6. Conclusão Final

A análise unificada demonstra que o **desmatamento não é o único motor do crescimento econômico** nas regiões estudadas. Enquanto alguns municípios atingem alto desenvolvimento humano e econômico de forma sustentável, outros permanecem em um ciclo de degradação sem retorno social efetivo. A fiscalização ambiental via embargos é um passo crucial, mas sua eficácia completa depende de mecanismos mais ágeis de regularização e fomento à produção eficiente.